# Video Restoration Master 2026 (v3)

Pipeline adaptativo para llevar video real degradado a un master 1080p con la mejor fidelidad posible.

**Cambios de la v3 sobre la v2** (tras una revisión de código y una revisión de arquitectura, ambas contrastadas contra fuentes reales antes de aceptarlas):
- Corrige pérdida de audio en el master.
- Corrige el bug real de `pick_batch_size` (ignoraba el límite de VRAM).
- Corrige el bug real de selección de modelo en A100/H100 (el texto decía 7B, el código usaba 3B).
- Comandos FFmpeg/Python vía `subprocess` con listas de argumentos (sin interpolar strings en shell).
- Nombre de archivo subido saneado.
- Lee metadata de color real (`color_space`/`color_primaries`/`color_transfer`) en vez de asumir BT.709 a ciegas.
- Detecta VFR vs CFR y evita forzar duplicado/eliminado de frames si no hace falta.
- Diagnóstico temporal ahora compara frames realmente consecutivos y compensa movimiento (optical flow) para no confundir movimiento de cámara con parpadeo del modelo.
- Clasificación CLEAN/NORMAL/SEVERE ahora también mira blur y bloques de compresión, no solo bitrate.
- Real-ESRGAN con tiling por GPU, se salta el modelo si el video ya es ≥1080p, y queda fijado al tag `v0.3.0`.
- SeedVR2 usa `--temporal_overlap`/`--prepend_frames`, verifica el código de salida y limpia resultados viejos antes de correr.

**Lo que NO se implementó (y por qué):**
- `--chunk_size`, `--10bit`, `--video_backend ffmpeg` como flags de SeedVR2: **no existen** en la documentación oficial del CLI, no puedo usar flags que no están ahí.
- STCDiT como motor: el paper es real (CVPR 2026) pero no tiene código/checkpoints públicos que yo haya podido encontrar.
- Benchmark automático SeedVR2 vs FlashVSR vs STCDiT: fuera de alcance por ahora.

### Antes de empezar
`Entorno de ejecución → Cambiar tipo de entorno de ejecución`: **A100** (recomendado) o **L4** (funciona bien con el modelo 3B FP8). T4 funciona pero es el escenario más lento/limitado.

Ejecuta las celdas **en orden**. Si alguna falla, copia el error completo tal cual.

## 0. Verificar GPU asignada

In [ ]:
import subprocess
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True).stdout.strip()
print('GPU detectada:', gpu_info if gpu_info else 'NINGUNA — ve a Entorno de ejecución > Cambiar tipo de entorno y elige una GPU.')

GPU_NAME = gpu_info.split(',')[0].strip() if gpu_info else ''
print('Nombre GPU:', GPU_NAME)

## 1a. Instalar Real-ESRGAN v0.3.0 (motor rápido / fallback)
Se fija al tag `v0.3.0` (última release, confirmada) en vez de seguir `main` sin versión, para que el notebook sea reproducible.

In [ ]:
import os, subprocess, glob

REALESRGAN_DIR = '/content/Real-ESRGAN'
if os.path.isdir(REALESRGAN_DIR):
    subprocess.run(['rm', '-rf', REALESRGAN_DIR], check=True)

subprocess.run(['git', 'clone', '-q', '--branch', 'v0.3.0', '--depth', '1',
                 'https://github.com/xinntao/Real-ESRGAN.git', REALESRGAN_DIR], check=True)

subprocess.run(['pip', 'install', '-q', 'numpy<2'], check=True)
subprocess.run(['pip', 'install', '-q', 'basicsr', 'facexlib', 'gfpgan'], check=True)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REALESRGAN_DIR, check=True)
subprocess.run(['python', 'setup.py', 'develop', '-q'], cwd=REALESRGAN_DIR, check=True)

# Parche conocido: basicsr/facexlib/gfpgan importan un módulo eliminado en torchvision >= 0.17.
# Esto es independiente de la versión de Real-ESRGAN (es un problema entre basicsr y torchvision),
# así que fijar la versión de Real-ESRGAN no lo evita — se sigue necesitando.
targets = (glob.glob('/usr/**/basicsr/**/*.py', recursive=True)
           + glob.glob('/usr/**/facexlib/**/*.py', recursive=True)
           + glob.glob('/usr/**/gfpgan/**/*.py', recursive=True))
patched = 0
for f in targets:
    try:
        content = open(f, encoding='utf-8').read()
    except Exception:
        continue
    if 'torchvision.transforms.functional_tensor' in content:
        open(f, 'w', encoding='utf-8').write(
            content.replace('torchvision.transforms.functional_tensor', 'torchvision.transforms.functional'))
        patched += 1
print(f'Parche aplicado a {patched} archivo(s).')

weights_dir = os.path.join(REALESRGAN_DIR, 'weights')
os.makedirs(weights_dir, exist_ok=True)
subprocess.run(['wget', '-q', '-nc', '-P', weights_dir,
                 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth'], check=True)

import importlib
import basicsr.data.degradations
importlib.reload(basicsr.data.degradations)
print('\n✅ Real-ESRGAN v0.3.0 listo.')

## 1b. Instalar SeedVR2 (motor principal, standalone CLI, sin ComfyUI)

In [ ]:
import os, subprocess

SEEDVR2_DIR = '/content/seedvr2_videoupscaler'
if os.path.isdir(SEEDVR2_DIR):
    subprocess.run(['rm', '-rf', SEEDVR2_DIR], check=True)

subprocess.run(['git', 'clone', '-q', 'https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git', SEEDVR2_DIR], check=True)

# Filtrar líneas de torch/torchvision/torchaudio: usamos el torch+CUDA que ya trae Colab
req_path = os.path.join(SEEDVR2_DIR, 'requirements.txt')
req_colab_path = os.path.join(SEEDVR2_DIR, 'requirements_colab.txt')
with open(req_path) as f:
    lines = f.readlines()
filtered = [l for l in lines if not l.strip().lower().startswith(('torch', 'torchvision', 'torchaudio'))]
with open(req_colab_path, 'w') as f:
    f.writelines(filtered)

subprocess.run(['pip', 'install', '-q', '-r', req_colab_path], check=True)

import torch
print('Torch:', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())

help_out = subprocess.run(['python', 'inference_cli.py', '--help'], cwd=SEEDVR2_DIR,
                           capture_output=True, text=True)
print(help_out.stdout[:1200])
print('\n✅ SeedVR2 CLI listo (los pesos del modelo se descargan automáticamente en la primera corrida).')
print('Si el bloque de --help de arriba no muestra flags como --temporal_overlap o --batch_size,')
print('el proyecto cambió su interfaz desde que se escribió este notebook y hay que ajustar la celda 6.')

## 2. Sube tu video
El nombre del archivo se sanea (solo letras/números/`._-`) antes de usarse en cualquier ruta o comando.

In [ ]:
from google.colab import files
import os, shutil, re

os.chdir('/content')
os.makedirs('/content/input', exist_ok=True)
os.makedirs('/content/work', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No se subió ningún archivo.')

raw_name = next(iter(uploaded.keys()))
safe_name = os.path.basename(raw_name)
safe_name = re.sub(r'[^A-Za-z0-9._-]', '_', safe_name)
if not safe_name.lower().endswith(('.mp4', '.mov', '.mkv', '.avi', '.webm')):
    raise ValueError(f'Formato de video no soportado: {safe_name}')

input_filename = safe_name
input_path = os.path.join('/content/input', input_filename)
shutil.move(raw_name, input_path)
print(f'Video subido: {input_path}')

## 3. Análisis del video (ffprobe)
Incluye metadata de color real (con fallback a BT.709 solo si no viene declarada) y detección de VFR/CFR.

In [ ]:
import subprocess, json as _json

def ffprobe_json(path):
    cmd = ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_format', '-show_streams', path]
    res = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return _json.loads(res.stdout)

probe = ffprobe_json(input_path)
vstreams = [s for s in probe['streams'] if s['codec_type'] == 'video']
if not vstreams:
    raise ValueError('El archivo no contiene una pista de video.')
vstream = vstreams[0]
astreams = [s for s in probe['streams'] if s['codec_type'] == 'audio']
fmt = probe['format']

width = int(vstream['width'])
height = int(vstream['height'])

def parse_rate(s, default=25.0):
    try:
        num, den = s.split('/')
        den = float(den)
        return float(num) / den if den else default
    except Exception:
        return default

r_fps = parse_rate(vstream.get('r_frame_rate', '25/1'))
avg_fps = parse_rate(vstream.get('avg_frame_rate', '25/1'))
fps = r_fps  # r_frame_rate es la tasa nominal, la usamos para el master
IS_VFR = abs(r_fps - avg_fps) > 0.05  # si difieren, probablemente hay variación real de frame rate

duration = float(fmt.get('duration', vstream.get('duration', 0)) or 0)
nb_frames = int(vstream.get('nb_frames', 0) or (duration * fps))
bitrate = int(fmt.get('bit_rate', 0) or vstream.get('bit_rate', 0) or 0)
codec = vstream.get('codec_name', 'desconocido')
pix_fmt = vstream.get('pix_fmt', 'desconocido')
has_audio = len(astreams) > 0

# Metadata de color real; si no viene declarada, se asume bt709 (razonable para material de celular/redes,
# pero ya no es una suposición ciega si el archivo sí trae la etiqueta)
color_space = vstream.get('color_space') or 'bt709'
color_primaries = vstream.get('color_primaries') or 'bt709'
color_transfer = vstream.get('color_transfer') or 'bt709'

print('--- Info del video ---')
print(f'Resolución:     {width}x{height}')
print(f'FPS (r/avg):    {r_fps:.3f} / {avg_fps:.3f}  {"(VFR detectado)" if IS_VFR else "(CFR)"}')
print(f'Duración:       {duration:.1f} s')
print(f'Frames (aprox): {nb_frames}')
print(f'Códec:          {codec}  |  pix_fmt: {pix_fmt}')
print(f'Color:          space={color_space}  primaries={color_primaries}  transfer={color_transfer}'
      + ('  (declarado en el archivo)' if vstream.get('color_space') else '  (no declarado, se asume bt709)'))
print(f'Bitrate:        {bitrate/1000:.0f} kbps' if bitrate else 'Bitrate:        no reportado, se estimará')
print(f'Audio:          {"sí" if has_audio else "no"} ({len(astreams)} pista(s))')

if not bitrate and duration > 0:
    size_bytes = int(fmt.get('size', 0) or 0)
    bitrate = int(size_bytes * 8 / duration) if size_bytes else 0
    print(f'Bitrate estimado por tamaño de archivo: {bitrate/1000:.0f} kbps')

## 4. Clasificación automática de calidad
Combina bits-por-píxel, resolución, una estimación de nitidez (varianza de Laplaciano) y una estimación simple de bloques de compresión JPEG/H.264 sobre unos pocos frames de muestra. **Sigue siendo una heurística, no un modelo de calidad perceptual entrenado** — es mejor que solo bitrate, pero no es infalible. Puedes sobreescribir `TIER` a mano.

In [ ]:
import cv2, numpy as np

def sample_frames(path, n=5):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    idxs = np.linspace(0, max(total - 1, 0), num=min(n, total), dtype=int)
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    cap.release()
    return frames

def blur_score(gray_frames):
    # varianza del Laplaciano: bajo = borroso, alto = nítido. Normalizado de forma aproximada.
    if not gray_frames:
        return None
    vals = [cv2.Laplacian(g, cv2.CV_64F).var() for g in gray_frames]
    return float(np.mean(vals))

def blockiness_score(gray_frames):
    # heurística simple: compara gradiente en fronteras de bloque de 8px vs el resto.
    # valores >1 sugieren bloques de compresión visibles (blocking artifact).
    if not gray_frames:
        return None
    ratios = []
    for g in gray_frames:
        gx = np.abs(np.diff(g.astype(np.float32), axis=1))
        cols = np.arange(gx.shape[1])
        boundary_mask = (cols % 8 == 7)
        boundary_mean = gx[:, boundary_mask].mean() if boundary_mask.any() else 0
        nonboundary_mean = gx[:, ~boundary_mask].mean() if (~boundary_mask).any() else 1e-6
        ratios.append(boundary_mean / max(nonboundary_mean, 1e-6))
    return float(np.mean(ratios))

sampled = sample_frames(input_path, n=5)
blur = blur_score(sampled)
blockiness = blockiness_score(sampled)

bpp = bitrate / (width * height * fps) if (width and height and fps and bitrate) else 0

print(f'Bits por píxel (bpp): {bpp:.4f}')
print(f'Nitidez (Laplaciano, más alto = más nítido): {blur:.1f}' if blur is not None else 'Nitidez: no calculada')
print(f'Bloques de compresión (más alto = más bloques visibles): {blockiness:.3f}' if blockiness is not None else 'Bloques: no calculado')

if bpp >= 0.12 and min(width, height) >= 480:
    TIER = 'CLEAN'
elif bpp >= 0.04:
    TIER = 'NORMAL'
else:
    TIER = 'SEVERE'

if min(width, height) < 360 and TIER == 'CLEAN':
    TIER = 'NORMAL'

# Ajuste por nitidez/bloques: si el bitrate sugería CLEAN pero el video está borroso o con bloques
# visibles, se degrada la clasificación un nivel.
BLUR_THRESHOLD = 60.0       # por debajo de esto se considera borroso (umbral aproximado, no calibrado formalmente)
BLOCKINESS_THRESHOLD = 1.3  # por encima de esto se considera con bloques visibles

degraded_by_visual = (blur is not None and blur < BLUR_THRESHOLD) or (blockiness is not None and blockiness > BLOCKINESS_THRESHOLD)
if degraded_by_visual and TIER == 'CLEAN':
    TIER = 'NORMAL'
    print('Ajuste: bitrate sugería CLEAN, pero el video se ve borroso/con bloques -> bajado a NORMAL.')
elif degraded_by_visual and TIER == 'NORMAL':
    TIER = 'SEVERE'
    print('Ajuste: NORMAL con señales visuales fuertes de degradación -> subido a SEVERE.')

print(f'\nClasificación final: {TIER}')
print('  CLEAN  -> Real-ESRGAN (rápido)')
print('  NORMAL -> SeedVR2 directo')
print('  SEVERE -> pre-limpieza ligera + SeedVR2')
print('\nPara forzar manualmente, descomenta y edita:')
# TIER = 'NORMAL'  # 'CLEAN' | 'NORMAL' | 'SEVERE'

## 5. Pre-limpieza ligera (solo si SEVERE)
Intermedio casi sin pérdida (`crf 0`) para no sumar una generación de compresión con pérdida justo antes de la restauración.

In [ ]:
import subprocess, shutil

work_input = '/content/work/pre_cleaned.mp4'

if TIER == 'SEVERE':
    print('Aplicando denoise ligero (hqdn3d), intermedio casi-lossless...')
    cmd = [
        'ffmpeg', '-y', '-i', input_path,
        '-vf', 'hqdn3d=1.5:1.5:3:3',
        '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '0',
        '-c:a', 'copy', work_input
    ]
    subprocess.run(cmd, check=True)
else:
    shutil.copy(input_path, work_input)
    print('TIER != SEVERE, se usa el video original sin pre-limpieza.')

print('Archivo de trabajo:', work_input)

## 6. Restauración / upscaling

`SEEDVR2_TRY_7B`: déjalo en `False` por defecto. El modelo 7B da más detalle pero la comunidad ha reportado artefactos de bandas verticales en algunas configuraciones — actívalo solo si quieres experimentar y comparar tú mismo.

In [ ]:
import math, os, glob, subprocess

TARGET_HEIGHT = 1080
SEEDVR2_TRY_7B = False  # True = experimental, con riesgo conocido de artefactos en algunos casos

restored_video_path = None

def pick_batch_size(n_frames, max_batch):
    candidates = [1, 5, 9, 13, 17, 21, 25, 33, 41]
    valid = [c for c in candidates if c <= max(n_frames, 1) and c <= max_batch]
    return valid[-1] if valid else 1

def gpu_tier(gpu_name):
    g = gpu_name.upper()
    if 'A100' in g or 'H100' in g:
        return 'HIGH'
    if 'L4' in g:
        return 'MID'
    return 'LOW'  # T4 u otra

TIER_MAX_BATCH = {'HIGH': 33, 'MID': 21, 'LOW': 9}

def pick_seedvr2_model(tier, try_7b):
    if tier == 'HIGH':
        if try_7b:
            print('⚠️  Usando SeedVR2 7B: puede dar más detalle, pero hay reportes de artefactos de bandas')
            print('    verticales en algunas configuraciones. Revisa el resultado con atención.')
            return 'seedvr2_ema_7b_fp16.safetensors', {}
        return 'seedvr2_ema_3b_fp16.safetensors', {}
    if tier == 'MID':
        return 'seedvr2_ema_3b_fp8_e4m3fn.safetensors', {}
    return 'seedvr2_ema_3b-Q8_0.gguf', {'blocks_to_swap': 24, 'dit_offload_device': 'cpu', 'swap_io_components': True}

tier = gpu_tier(GPU_NAME)
max_batch = TIER_MAX_BATCH[tier]

if TIER == 'CLEAN':
    print('Motor: Real-ESRGAN (realesr-general-x4v3)')
    if height >= TARGET_HEIGHT:
        print(f'El video ya tiene {height}px de alto (>= {TARGET_HEIGHT}), se omite el upscaling con IA.')
        restored_video_path = work_input
    else:
        outscale = TARGET_HEIGHT / height
        tile = {'LOW': 256, 'MID': 512, 'HIGH': 0}[tier]
        print(f'Escala calculada: {outscale:.3f}x  ({width}x{height} -> ~{int(width*outscale)}x{TARGET_HEIGHT})  tile={tile or "auto/off"}')
        cmd = [
            'python', 'inference_realesrgan_video.py',
            '-i', work_input,
            '-n', 'realesr-general-x4v3',
            '-s', '4',
            '--outscale', f'{outscale:.4f}',
            '-dn', '0.4',
            '-o', '/content/work',
            '--suffix', 'restored'
        ]
        if tile:
            cmd += ['--tile', str(tile), '--tile_pad', '10']
        res = subprocess.run(cmd, cwd=REALESRGAN_DIR, capture_output=True, text=True)
        if res.returncode != 0:
            print('STDOUT:', res.stdout[-2000:])
            print('STDERR:', res.stderr[-2000:])
            raise RuntimeError(f'Real-ESRGAN falló con código {res.returncode}')
        matches = sorted(glob.glob('/content/work/*restored*.mp4'))
        if not matches:
            raise FileNotFoundError('Real-ESRGAN no generó ningún MP4 en /content/work')
        restored_video_path = matches[0]
    print('Salida:', restored_video_path)

else:
    dit_model, extra = pick_seedvr2_model(tier, SEEDVR2_TRY_7B)
    batch_size = pick_batch_size(nb_frames, max_batch)
    temporal_overlap = 3 if batch_size >= 9 else 0
    prepend_frames = 4
    print(f'Motor: SeedVR2  |  modelo: {dit_model}  |  GPU tier: {tier}  |  batch_size: {batch_size} (max {max_batch})')

    restored_frames_dir = '/content/work/seedvr2_out'
    if os.path.isdir(restored_frames_dir):
        import shutil as _shutil
        _shutil.rmtree(restored_frames_dir)
    os.makedirs(restored_frames_dir, exist_ok=True)

    extra_flags = []
    if extra.get('blocks_to_swap'):
        extra_flags += ['--blocks_to_swap', str(extra['blocks_to_swap'])]
    if extra.get('dit_offload_device'):
        extra_flags += ['--dit_offload_device', extra['dit_offload_device']]
    if extra.get('swap_io_components'):
        extra_flags += ['--swap_io_components']

    cmd = ['python', 'inference_cli.py', work_input,
           '--output', restored_frames_dir,
           '--output_format', 'mp4',
           '--dit_model', dit_model,
           '--resolution', str(TARGET_HEIGHT),
           '--batch_size', str(batch_size),
           '--uniform_batch_size',
           '--temporal_overlap', str(temporal_overlap),
           '--prepend_frames', str(prepend_frames),
           '--color_correction', 'lab'] + extra_flags
    print('Comando:', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=SEEDVR2_DIR, capture_output=True, text=True)
    if res.returncode != 0:
        print('STDOUT:', res.stdout[-3000:])
        print('STDERR:', res.stderr[-3000:])
        raise RuntimeError(
            f'SeedVR2 falló con código {res.returncode}. Si el mensaje menciona un flag no reconocido, '
            'la CLI cambió de interfaz desde que se escribió este notebook — revisa la salida de --help del paso 1b.')

    matches = sorted(glob.glob(f'{restored_frames_dir}/*.mp4'))
    if not matches:
        raise FileNotFoundError(f'SeedVR2 no generó ningún MP4 en {restored_frames_dir}')
    restored_video_path = matches[0]
    print('Salida:', restored_video_path)

## 7. Diagnóstico de estabilidad temporal (informativo)
Compara frames **realmente consecutivos** (no muestreados a saltos) y **compensa el movimiento** con optical flow antes de medir la diferencia — así no se confunde el movimiento normal de cámara/objetos con parpadeo introducido por el modelo. Sigue siendo un diagnóstico, no corrige nada por sí solo.

In [ ]:
import cv2, numpy as np

def temporal_stability_score(path, sample_pairs=25):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 2:
        cap.release()
        return None

    n = min(sample_pairs, total - 1)
    idxs = np.linspace(0, total - 2, num=n, dtype=int)
    residuals = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret1, f1 = cap.read()
        ret2, f2 = cap.read()
        if not (ret1 and ret2):
            continue
        g1 = cv2.cvtColor(f1, cv2.COLOR_BGR2GRAY)
        g2 = cv2.cvtColor(f2, cv2.COLOR_BGR2GRAY)
        # compensación de movimiento: alinea g2 hacia g1 usando optical flow antes de comparar
        flow = cv2.calcOpticalFlowFarneback(g1, g2, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        h, w = g1.shape
        grid_x, grid_y = np.meshgrid(np.arange(w), np.arange(h))
        map_x = (grid_x + flow[..., 0]).astype(np.float32)
        map_y = (grid_y + flow[..., 1]).astype(np.float32)
        g2_warped = cv2.remap(g2, map_x, map_y, cv2.INTER_LINEAR)
        residuals.append(np.mean(cv2.absdiff(g1, g2_warped)))
    cap.release()
    return float(np.mean(residuals)) if residuals else None

if restored_video_path:
    score = temporal_stability_score(restored_video_path)
    if score is not None:
        print(f'Residual medio tras compensar movimiento: {score:.2f} (escala 0-255)')
        print('Referencia orientativa (no calibrada formalmente): <5 estable, 5-10 normal, >10 revisa visualmente si hay parpadeo.')
    else:
        print('No se pudo calcular (video muy corto o no se generó salida).')
else:
    print('No hay video restaurado para analizar.')

### (Opcional) Aplicar deflicker si el diagnóstico salió alto

In [ ]:
import subprocess

APPLY_DEFLICKER = False  # cambia a True si el diagnóstico anterior salió alto y ves parpadeo

if APPLY_DEFLICKER and restored_video_path:
    deflickered = '/content/work/deflickered.mp4'
    cmd = ['ffmpeg', '-y', '-i', restored_video_path,
           '-vf', 'deflicker=mode=pm',
           '-c:v', 'libx264', '-preset', 'fast', '-crf', '12',
           '-c:a', 'copy', deflickered]
    subprocess.run(cmd, check=True)
    restored_video_path = deflickered
    print('Deflicker aplicado:', restored_video_path)
else:
    print('Deflicker no aplicado.')

## 8. Master 10-bit BT.709 (ProRes 422 HQ)
Usa la metadata de color real detectada en el paso 3 (no asume BT.709 a ciegas), y **reincorpora el audio desde el archivo original** (segundo input), no desde el video restaurado — que puede no traer audio.

In [ ]:
import subprocess

master_path = '/content/output/master_1080p_bt709.mov'

cmd = ['ffmpeg', '-y', '-i', restored_video_path]
if has_audio:
    cmd += ['-i', input_path, '-map', '0:v:0', '-map', '1:a:0?', '-c:a', 'pcm_s16le', '-shortest']
else:
    cmd += ['-map', '0:v:0', '-an']

vf = (f"zscale=matrixin={color_space}:matrix=bt709:"
      f"primariesin={color_primaries}:primaries=bt709:"
      f"transferin={color_transfer}:transfer=bt709,format=yuv422p10le")
cmd += ['-vf', vf,
        '-color_primaries', 'bt709', '-color_trc', 'bt709', '-colorspace', 'bt709',
        '-c:v', 'prores_ks', '-profile:v', '3']

if IS_VFR:
    cmd += ['-fps_mode', 'passthrough']
    print('VFR detectado: se preservan los timestamps originales (passthrough) en vez de forzar CFR.')
else:
    cmd += ['-r', str(fps)]

cmd += [master_path]
subprocess.run(cmd, check=True)

print('Master generado:', master_path)

## 9. Copia de entrega (H.264, para compartir/ver)

In [ ]:
import subprocess

delivery_path = '/content/output/delivery_1080p_h264.mp4'

cmd = ['ffmpeg', '-y', '-i', master_path,
       '-c:v', 'libx264', '-preset', 'slow', '-crf', '17', '-pix_fmt', 'yuv420p',
       '-c:a', 'aac', '-b:a', '256k',
       '-movflags', '+faststart',
       delivery_path]
subprocess.run(cmd, check=True)

print('Entrega generada:', delivery_path)

## 10. Descargar resultados
El master ProRes puede ser muy pesado (varios GB). Si falla la descarga directa por tamaño, monta Google Drive y cópialo ahí.

In [ ]:
from google.colab import files
import os

print('Tamaño delivery (MB):', round(os.path.getsize(delivery_path)/1e6, 1))
print('Tamaño master (MB):  ', round(os.path.getsize(master_path)/1e6, 1))

files.download(delivery_path)
# Descomenta si también quieres bajar el master pesado:
# files.download(master_path)

In [ ]:
# Opcional: guardar el master en Google Drive en vez de descargarlo por el navegador
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(master_path, '/content/drive/MyDrive/master_1080p_bt709.mov')

## Notas y límites honestos

- La clasificación CLEAN/NORMAL/SEVERE combina bitrate + nitidez + bloques, pero **sigue siendo heurística**, no un modelo de calidad perceptual entrenado. Los umbrales (`BLUR_THRESHOLD`, `BLOCKINESS_THRESHOLD`) son aproximados — si el resultado no tiene sentido para tu video, ajusta `TIER` a mano en el paso 4.
- El diagnóstico temporal del paso 7 ahora compensa movimiento, pero sigue siendo una **medida aproximada**, no un análisis de flicker validado formalmente. `deflicker` es una herramienta genérica de ffmpeg, no específica de IA.
- **SeedVR2 es un proyecto activo** y su CLI ha cambiado de interfaz antes (versión 2.5 fue un *breaking change*). Si el paso 1b muestra un `--help` distinto a los flags usados en el paso 6, hay que ajustar los comandos.
- El modelo **7B de SeedVR2 es experimental en este notebook** (`SEEDVR2_TRY_7B = False` por defecto): hay reportes de la comunidad de artefactos de bandas verticales en algunas configuraciones. Actívalo si quieres comparar, pero revisa el resultado.
- **VRAM**: en GPUs con poca memoria (T4) puede fallar con OOM en videos largos incluso con GGUF+BlockSwap — en ese caso, corta el video en segmentos más cortos con ffmpeg antes de subirlo.
- Si prefieres DNxHR en vez de ProRes para el master (paso 8), cambia `'-c:v', 'prores_ks', '-profile:v', '3'` por `'-c:v', 'dnxhd', '-profile:v', 'dnxhr_hq', '-pix_fmt', 'yuv422p10le'`.
- No se integró STCDiT (sin código público disponible) ni un benchmark automático SeedVR2/FlashVSR/STCDiT — quedan como posible trabajo futuro si lo pides explícitamente.